In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga del shapefile de secciones censales

In [2]:
# Cargo el GDF con las secciones censales
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)

# Compruebo que se haya cargado
print(gdf_secciones.info())
gdf_secciones.sample(5)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   CUSEC     3535 non-null   object  
 1   CUMUN     3535 non-null   object  
 2   CSEC      3535 non-null   object  
 3   CDIS      3535 non-null   object  
 4   CMUN      3535 non-null   object  
 5   CPRO      3535 non-null   object  
 6   CCA       3535 non-null   object  
 7   CUDIS     3535 non-null   object  
 8   CLAU2     3535 non-null   object  
 9   NPRO      3535 non-null   object  
 10  NCA       3535 non-null   object  
 11  CNUT0     3535 non-null   object  
 12  CNUT1     3535 non-null   object  
 13  CNUT2     3535 non-null   object  
 14  CNUT3     3535 non-null   object  
 15  NMUN      3535 non-null   object  
 16  geometry  3535 non-null   geometry
dtypes: geometry(1), object(16)
memory usage: 469.6+ KB
None


,CUSEC,CUMUN,CSEC,CDIS,CMUN,CPRO,CCA,CUDIS,CLAU2,NPRO,NCA,CNUT0,CNUT1,CNUT2,CNUT3,NMUN,geometry
2746,4710101001,47101,001,01,101,47,07,4710101,47101,Valladolid,Castilla y León,ES,4,1,8,Nava del Rey,"POLYGON ((329741.873 4591672.135, 329709.875 4..."
1644,3706901001,37069,001,01,069,37,07,3706901,37069,Salamanca,Castilla y León,ES,4,1,5,Calvarrasa de Abajo,"POLYGON ((287196.427 4538767.316, 287241.426 4..."
1957,3727406005,37274,005,06,274,37,07,3727406,37274,Salamanca,Castilla y León,ES,4,1,5,Salamanca,"POLYGON ((276713.964 4539178.691, 276760.218 4..."
3088,4718611032,47186,032,11,186,47,07,4718611,47186,Valladolid,Castilla y León,ES,4,1,8,Valladolid,"POLYGON ((354003.21 4608663.607, 354019.254 46..."
767,0932801001,09328,001,01,328,09,07,0932801,09328,Burgos,Castilla y León,ES,4,1,2,Rucandio,"POLYGON ((459446.075 4736107.723, 459826.084 4..."


# Calculo de la superficie de la sección censal

In [3]:
# Normalizamos campos clave
gdf_secciones["CUSEC"] = gdf_secciones["CUSEC"].astype(str).str.strip().str.zfill(10)
gdf_secciones["CUMUN"] = gdf_secciones["CUMUN"].astype(str).str.strip().str.zfill(5)
gdf_secciones["NPRO"] = gdf_secciones["NPRO"].str.strip()

# Calculamos el área de cada sección (en km²)
gdf_secciones["area_km2"] = gdf_secciones.geometry.area / 1e6

gdf_secciones = gdf_secciones.rename(columns={
    "NPRO": "Provincia",
    "CUMUN": "CMuni"   # a veces viene como CUMUN o CMUNI, según fuente
})

gdf_secciones["Provincia"] = gdf_secciones["Provincia"].apply(normalize_text)

# --- 1️⃣ Área por sección censal ---
df_area_seccion = gdf_secciones[["Provincia", "CMuni", "CUSEC", "area_km2"]].copy()

# --- 2️⃣ Área total por municipio ---
df_area_municipio = (
    gdf_secciones
    .groupby(["Provincia", "CMuni"], as_index=False)
    .agg(area_km2=("area_km2", "sum"))
)

# --- 3️⃣ Área total por provincia ---
df_area_provincia = (
    gdf_secciones
    .groupby("Provincia", as_index=False)
    .agg(area_km2=("area_km2", "sum"))
)

# Mostramos resultados de control
print(f"✅ Secciones procesadas: {len(df_area_seccion):,}")
print(f"🏙️ Municipios calculados: {len(df_area_municipio):,}")
print(f"🌄 Provincias calculadas: {len(df_area_provincia):,}")

✅ Secciones procesadas: 3,535
🏙️ Municipios calculados: 2,248
🌄 Provincias calculadas: 9


In [4]:
print(df_area_seccion.info())
print(df_area_seccion.sample(5))
print("\n-------------------------------------\n")
print(df_area_municipio.info())
print(df_area_municipio.sample(5))
print("\n-------------------------------------\b")
print(df_area_provincia.info())
print(df_area_provincia.sample(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Provincia  3535 non-null   object 
 1   CMuni      3535 non-null   object 
 2   CUSEC      3535 non-null   object 
 3   area_km2   3535 non-null   float64
dtypes: float64(1), object(3)
memory usage: 110.6+ KB
None
       Provincia  CMuni       CUSEC   area_km2
3450      Zamora  49275  4927502015   0.348604
1737   Salamanca  37156  3715601003  42.139450
1637   Salamanca  37061  3706101001  47.392538
3050  Valladolid  47186  4718610046   0.258381
1005        Leon  24089  2408903002   0.116686

-------------------------------------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2248 entries, 0 to 2247
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Provincia  2248 non-null   object 
 1   CMuni      2248 non-null   object 
 2   ar

In [5]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DD, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DD, "superficie_seccion.csv")
ruta_municipio = os.path.join(DATA_OUTPUTS_DD, "superficie_municipio.csv")
ruta_provincia = os.path.join(DATA_OUTPUTS_DD, "superficie_provincia.csv")

# Guardar DataFrames
df_area_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
df_area_municipio.to_csv(
    ruta_municipio,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
df_area_provincia.to_csv(
    ruta_provincia,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DD}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DD_Dim_demografica
